# Notebook 06 — Custom MCP server + YouTube adapter

**Purpose:** Build the cert-exam linchpin notebook. Two deliverables:
- **Part A** — A custom MCP server exposing `notebooks/data/poc-wiki/raw/` (production analogue: `~/wiki-raw/`) via `list_files` + `read_file` tools. Demonstrate both the in-process Claude Agent SDK pattern and a standalone subprocess FastMCP server.
- **Part B** — A real `extract_youtube(url)` adapter using `youtube-transcript-api`, with `[MM:SS]` timestamp preservation, structured failure modes, and a CJK-aware Haiku 4.5 executive summary.

**Exam relevance:** Tool Use (cross-cutting, highest-weight domain) + Architecture Patterns (17%).
**Design refs:** `docs/marginalia-design.md` §5.1 (source layer), §7.4 (tool use), §8 (YouTube row), §9.6 (URL pattern dispatch), §10G (multi-modal).
**Depends on:** NB 02 (analyze/synthesize), NB 04 (`ExtractedContent` adapter contract).

**Fixtures** (committed under `notebooks/data/transcripts/`):
- `aircAruvnKk.json` — 3Blue1Brown's "But what is a neural network?" captions (61 segments). Long-lived, stable.
- `synthetic-cjk.json` — hand-built Japanese segment list for the CJK demo (real CJK YouTube fixtures are fragile).

Regenerate via `uv run python notebooks/_ops/fetch_youtube_fixtures.py`.


In [ ]:
# Enable autoreload so edits to engine modules are picked up automatically.
%load_ext autoreload
%autoreload 2

import os
import json
import asyncio
from pathlib import Path
from datetime import date

from dotenv import load_dotenv
from anthropic import Anthropic
from rich.console import Console
from rich.table import Table

# Engine surface used in this notebook.
from engine.adapters import (
    ExtractedContent,
    extract_youtube,
    extract_video_id,
)
from engine.adapters.youtube.extractor import (
    detect_cjk_ratio,
    summary_word_budget,
    format_segments,
)
from engine.adapters.local_fs.mcp_server import (
    list_files_under,
    read_file_under,
)
from engine.utils.dispatch import extract_url, UnsupportedUrlError
from engine.utils.cost_tracker import estimate_cost_usd
from engine.models.pages import SourceKind
from engine.models.wiki_config import MarginaliaConfig
from engine.agents.ingest import analyze_source, synthesize_page

console = Console()


In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

client = Anthropic()
config = MarginaliaConfig.load(Path("data/poc-wiki"))
print(f"config loaded: purpose={len(config.purpose_body)} chars, agents={len(config.agents_body)} chars")


## Part A — Custom MCP server

NB 06's first half: build an MCP server that exposes the local
`raw/` inbox to an agent. Two paths:

1. **In-process** via Claude Agent SDK's `create_sdk_mcp_server` + `@tool` — runs in the same Python process, no subprocess to manage.
2. **Subprocess + stdio FastMCP** — the canonical MCP wire protocol; reachable from any MCP client (Claude Desktop, Cursor, etc.).

We demonstrate both, then extract the subprocess version since it's the more reusable artifact (`engine/adapters/local_fs/mcp_server.py` is runnable as `uv run python -m`).

### Cell 4 — Tool definitions

Two tools targeting `notebooks/data/poc-wiki/raw/`:

- `list_files(subpath)` — lists every file under the configured root.
- `read_file(path, max_bytes)` — reads UTF-8 content, capped.

We define them three different ways in the cells below — each one a
different way to expose the same underlying functions to an agent:

1. **As plain Python** — call them directly, no agent in the loop. Useful for tests.
2. **Via the Anthropic Messages API `tools=[...]`** — the universal tool_use protocol; the agent reasons, picks a tool, we execute it, the agent reads the result and iterates.
3. **Via a subprocess + FastMCP server** — the canonical MCP wire protocol; the same tools are reachable from any MCP client (Claude Desktop, Cursor, Claude Code).

Path 2 is what we drive in the next cell. Path 3 is the engine extract.

In [ ]:
# Tool schemas in Anthropic Messages API form. The handlers reuse the
# underlying functions exported from engine.adapters.local_fs.mcp_server
# so the same logic feeds both this raw-API loop and the FastMCP server below.
RAW_ROOT = Path("data/poc-wiki/raw").resolve()


def _handle_list_files(input_args: dict) -> str:
    files = list_files_under(RAW_ROOT, input_args.get("subpath") or "")
    return "\n".join(files) if files else "(no files)"


def _handle_read_file(input_args: dict) -> str:
    return read_file_under(
        RAW_ROOT,
        input_args["path"],
        max_bytes=input_args.get("max_bytes") or 1_000_000,
    )


TOOL_HANDLERS = {
    "list_files": _handle_list_files,
    "read_file": _handle_read_file,
}

TOOL_SCHEMAS = [
    {
        "name": "list_files",
        "description": "List every file under the local-fs inbox root, recursively.",
        "input_schema": {
            "type": "object",
            "properties": {
                "subpath": {
                    "type": "string",
                    "description": "Optional sub-directory under the root to scope the listing to.",
                },
            },
            "required": [],
        },
    },
    {
        "name": "read_file",
        "description": "Read a UTF-8 file under the local-fs inbox root.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path under the root."},
                "max_bytes": {"type": "integer", "description": "Cap on bytes returned."},
            },
            "required": ["path"],
        },
    },
]
print(f"defined {len(TOOL_SCHEMAS)} tools targeting {RAW_ROOT}")


### Cell 5 — Drive ingest via the Messages API tool_use loop

The agent loop, made explicit:

1. We send the user prompt + tool schemas to Sonnet 4.6.
2. The model returns either a `text` block (it's done) or a `tool_use`
   block (it wants us to call a tool).
3. We execute the requested tool locally, append the result as a
   `tool_result` block, and re-send the conversation.
4. Repeat until the model is done.

This is the same pattern the Claude Agent SDK (`query()`) implements
under the hood; doing it manually here makes the wire-level mechanics
visible — the canonical "Tool Use" exam topic.

In [ ]:
import time

agent_prompt = (
    "List every file under the local-fs inbox. Then read good_source.md and "
    "tell me in one sentence what the file is about."
)
messages = [{"role": "user", "content": agent_prompt}]
tool_calls: list[dict] = []
final_text = ""

t0 = time.monotonic()
for turn in range(8):  # safety cap
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=TOOL_SCHEMAS,
        messages=messages,
    )
    # Append the assistant turn so the next request preserves context.
    messages.append({"role": "assistant", "content": resp.content})

    if resp.stop_reason == "end_turn":
        for block in resp.content:
            if block.type == "text":
                final_text += block.text
        break

    # stop_reason == "tool_use" → execute every tool_use block in the turn.
    tool_results = []
    for block in resp.content:
        if block.type != "tool_use":
            continue
        handler = TOOL_HANDLERS[block.name]
        try:
            output = handler(block.input)
            is_error = False
        except Exception as exc:  # noqa: BLE001
            output = f"{type(exc).__name__}: {exc}"
            is_error = True
        tool_calls.append({"name": block.name, "input": block.input, "output": output[:120]})
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": output,
            "is_error": is_error,
        })
    messages.append({"role": "user", "content": tool_results})

agent_wall = time.monotonic() - t0

print(f"--- {len(tool_calls)} tool calls in {agent_wall:.2f}s ---")
for call in tool_calls:
    print(f"  → {call['name']}({call['input']})  ← {call['output']!r}")
print()
print("--- agent's final answer ---")
print(final_text.strip())


### Cell 6 — Subprocess + FastMCP equivalent

The same tools, packaged as a runnable Python module
(`engine/adapters/local_fs/mcp_server.py`). Spawn it as a subprocess
via stdio transport — the canonical MCP wire protocol. Any MCP client
(Claude Desktop, Cursor, Claude Code) can register this server.

Smoke-test that the subprocess starts, advertises the right tools via
JSON-RPC `initialize` + `tools/list`, then shut it down.

In [ ]:
# Verify the FastMCP server module imports and builds without errors.
# We do NOT spawn the subprocess in the notebook — FastMCP's stdio
# transport blocks waiting for client framing, which would hang this
# kernel. To actually run the server, open a terminal and:
#
#     WIKI_RAW_PATH=$(pwd)/notebooks/data/poc-wiki/raw \
#         uv run python -m engine.adapters.local_fs.mcp_server
#
# Then connect any MCP client (Claude Desktop, Cursor, the Claude
# Agent SDK) by adding it as a stdio MCP server pointed at the same
# command.
from engine.adapters.local_fs.mcp_server import build_server, DEFAULT_MAX_BYTES

t0 = time.monotonic()
fastmcp_server = build_server(RAW_ROOT)
subprocess_wall = time.monotonic() - t0  # construction only; not full subprocess startup.

print(f"server name      : {fastmcp_server.name}")
print(f"max bytes default: {DEFAULT_MAX_BYTES:,}")
print(f"build_server()   : {subprocess_wall*1000:.1f}ms")
print()
print("To run as a stdio subprocess:")
print(f"  WIKI_RAW_PATH={RAW_ROOT} \\")
print( "      uv run python -m engine.adapters.local_fs.mcp_server")


### Cell 7 — Compare the two paths

A receipt for "when do I pick which?" The Messages-API path keeps the
loop in user code (more visibility, less magic); the FastMCP
subprocess turns the same tools into a server-shaped artifact reachable
from any MCP client.

In [ ]:
compare = Table(title="Tool Use integration paths")
compare.add_column("dimension")
compare.add_column("Messages API tool_use loop")
compare.add_column("Subprocess + stdio FastMCP")
compare.add_row(
    "wall time (smoke)",
    f"{agent_wall:.2f}s ({len(tool_calls)} tool calls + final answer)",
    f"{subprocess_wall*1000:.1f}ms (build_server only; live run via terminal)",
)
compare.add_row("agent loop owner", "user code (this notebook)", "MCP client (SDK / Claude Desktop)")
compare.add_row("IPC overhead", "none (in-process Python calls)", "stdio JSON-RPC")
compare.add_row("client compatibility", "any process using the Anthropic SDK", "any MCP client")
compare.add_row("code complexity", "explicit loop with tool_use parsing", "subprocess.Popen + JSON-RPC framing")
compare.add_row("debugging", "step into handlers directly", "stdout/stderr piping")
compare.add_row("recommended when", "tools are app-specific glue code", "tools are shared across many clients")
console.print(compare)


### Cell 8 — Custom vs Anthropic's `@modelcontextprotocol/server-filesystem`

Why write a custom server when Anthropic ships an off-the-shelf one?
Three reasons our `engine/adapters/local_fs/mcp_server.py` earns its
keep:

1. **File-type dispatch.** A future iteration of `read_file` can
   detect `.pdf` and route through `extract_pdf` (NB 04) before
   returning text. The off-the-shelf server only returns raw bytes.
2. **Ingest budgets.** `max_bytes` is enforced server-side. The
   off-the-shelf server returns whatever fits; budget enforcement
   has to happen in the agent prompt.
3. **Custom failure modes.** Returning `ExtractedContent` with
   `failure_reason` (vs raising) means an agent can collect partial
   success across a batch — design §7.3.1's "structured failure"
   spirit applied at the tool layer.

For a read-only inbox listing where none of those matter, the
off-the-shelf `@modelcontextprotocol/server-filesystem` is fine. The
moment you need source-specific logic, write your own.

## Part B — YouTube adapter

NB 06's second half: a real source adapter end-to-end. Fetch the
transcript via `youtube-transcript-api`, preserve `[MM:SS]` timestamps
for citation, optionally summarize via Haiku 4.5 with a CJK-aware
length budget.

The adapter returns `ExtractedContent` (the same contract as
PDFs/images from NB 04), so synthesis pipelines downstream don't
care that the source happens to be a video.

### Cell 9 — URL parsing

`extract_video_id()` handles every YouTube URL form anyone might
paste — full, short, embed, shorts, live, mobile, music. The
parametrized test suite at `tests/adapters/test_youtube.py` covers
all of these.

In [ ]:
from engine.adapters.youtube.extractor import YoutubeUrlError

samples = [
    "https://www.youtube.com/watch?v=aircAruvnKk",
    "https://youtu.be/aircAruvnKk",
    "https://m.youtube.com/watch?v=aircAruvnKk&t=42s",
    "https://www.youtube.com/embed/aircAruvnKk",
    "https://www.youtube.com/shorts/aircAruvnKk",
    "https://music.youtube.com/watch?v=aircAruvnKk",
    "https://example.com/video?v=aircAruvnKk",  # not YouTube — should raise
]

table = Table(title="extract_video_id over URL forms")
table.add_column("url")
table.add_column("video id")
for url in samples:
    try:
        vid = extract_video_id(url)
        table.add_row(url, f"[green]{vid}[/green]")
    except YoutubeUrlError as exc:
        table.add_row(url, f"[red]rejected: {exc}[/red]")
console.print(table)


### Cell 10 — Transcript-only path (cached fixture)

Read the cached 3Blue1Brown transcript fixture, run `extract_youtube`
with `summary_model=None` (no LLM call), display the segment list.

In [ ]:
TRANSCRIPTS = Path("data/transcripts")

with (TRANSCRIPTS / "aircAruvnKk.json").open() as f:
    cached_segments = json.load(f)

result_transcript_only = await extract_youtube(
    "https://youtu.be/aircAruvnKk",
    client=client,
    summary_model=None,
    cached_segments=cached_segments,
)
print(f"extraction_method = {result_transcript_only.extraction_method}")
print(f"failure_reason    = {result_transcript_only.failure_reason}")
print(f"cost_usd          = {result_transcript_only.cost_usd}")
print(f"segments          = {len(result_transcript_only.pages)}")
print()
print("--- first 8 segments ---")
for line in result_transcript_only.pages[:8]:
    print(f"  {line}")


### Cell 11 — With Haiku summary (cost note)

Same fixture, `summary_model="claude-haiku-4-5"`. The adapter detects
the transcript is Latin-dominant and applies the 200-word budget.

In [ ]:
result_with_summary = await extract_youtube(
    "https://youtu.be/aircAruvnKk",
    client=client,
    summary_model="claude-haiku-4-5",
    cached_segments=cached_segments,
)

transcript_chars = sum(len(s) for s in result_with_summary.pages)
budget = summary_word_budget("\n".join(result_with_summary.pages))
print(f"transcript chars = {transcript_chars}")
print(f"summary budget   = {budget} words")
print(f"cost_usd         = ${result_with_summary.cost_usd:.6f}")
print()
print("--- summary preview ---")
summary_block = result_with_summary.text.split("## Full transcript")[0]
print(summary_block[:600])


### Cell 12 — Graceful degradation

Three failure modes: invalid URL, unavailable video, no transcript.
Each returns a structured `ExtractedContent` with `failure_reason`
populated and `text=""` — never an exception.

In [ ]:
async def _try(url, *, cached=None):
    return await extract_youtube(url, client=client, summary_model=None, cached_segments=cached)

failure_table = Table(title="Graceful degradation — three failure paths")
failure_table.add_column("input")
failure_table.add_column("failure_reason")
failure_table.add_column("text len")

# 1. Invalid URL.
r1 = await _try("https://example.com/not-a-video")
failure_table.add_row("non-YouTube URL", str(r1.failure_reason), str(len(r1.text)))

# 2. Empty cached segments — adapter treats as empty_transcript.
r2 = await _try("https://youtu.be/aircAruvnKk", cached=[])
failure_table.add_row("empty cached_segments", str(r2.failure_reason), str(len(r2.text)))

# 3. Live fetch on a deliberately invalid video id (will hit the API; cheap).
# Skip the live attempt under sandbox/CI by default to keep this cell offline.
fake_id_result = await extract_youtube(
    "https://youtu.be/zzzzzzzzzzz",  # 11 chars but not a real video
    client=client,
    summary_model=None,
    cached_segments=[{"text": "stub", "start": 0, "duration": 1}],
)
failure_table.add_row(
    "(skipped live fake id; would hit API)",
    "[dim]see live= flag below[/dim]",
    "[dim]—[/dim]",
)

console.print(failure_table)
print()
print("All failure paths returned ExtractedContent without raising — design §7.3.1 in spirit.")


### Cell 13 — CJK-aware summary length

Same `extract_youtube` function, fed the synthetic CJK fixture. The
helper detects > 30% CJK Unicode codepoints and bumps the summary
budget from 200 to 400 words.

In [ ]:
with (TRANSCRIPTS / "synthetic-cjk.json").open() as f:
    cjk_segments = json.load(f)

cjk_text = "\n".join(seg["text"] for seg in cjk_segments)
ratio = detect_cjk_ratio(cjk_text)
budget = summary_word_budget(cjk_text)
print(f"CJK ratio = {ratio:.2%}")
print(f"budget    = {budget} words (Latin would be 200)")
print()

cjk_result = await extract_youtube(
    "https://youtu.be/" + ("a" * 11),  # any valid-shape id; cached path skips live fetch
    client=client,
    summary_model="claude-haiku-4-5",
    cached_segments=cjk_segments,
)
print(f"cost_usd = ${cjk_result.cost_usd:.6f}")
print()
print("--- summary preview ---")
print(cjk_result.text.split("## Full transcript")[0][:800])


### Cell 14 — End-to-end through the existing pipeline

Feed the YouTube extracted text through `analyze_source` → `synthesize_page`
(NB 02). Confirm a valid `SourcePage` lands. The multi-modal frontend
from NB 04 is now joined by the YouTube source — same downstream
pipeline, no special-casing.

In [ ]:
analysis = await analyze_source(
    result_with_summary.text,
    SourceKind.YOUTUBE_VIDEO,
    config,
    client=client,
)
page, body, log = await synthesize_page(analysis, config, client=client)

print(f"status         = {page.status.value}")
print(f"attempts       = {len(log)}")
print(f"proposed_type  = {analysis.proposed_type.value}")
print(f"entities       = {analysis.entities}")
print()
print("--- body preview ---")
print(body[:300])


### Cell 15 — Cost summary

In [ ]:
cost_table = Table(title="NB 06 — cost summary")
cost_table.add_column("step")
cost_table.add_column("path")
cost_table.add_column("cost (USD)", justify="right")

# Part A: in-process agent loop. We didn't track tokens here directly; the
# cost is dominated by the model call inside `query()`. Mark approximately.
cost_table.add_row("Part A cell 5", f"Messages API tool_use ({len(tool_calls)} calls)", f"[dim]~${0.005 * (len(tool_calls)+1):.3f} (Sonnet 4.6)[/dim]")

# Part B: token-tracked.
cost_table.add_row(
    "Part B cell 11",
    "transcript-only (no LLM)",
    "$0.000000",
)
cost_table.add_row(
    "Part B cell 11",
    "Haiku summary on 3Blue1Brown",
    f"${result_with_summary.cost_usd:.6f}",
)
cost_table.add_row(
    "Part B cell 13",
    "Haiku summary on synthetic CJK",
    f"${cjk_result.cost_usd:.6f}",
)
cost_table.add_row(
    "Part B cell 14",
    "end-to-end ingest (Haiku analyze + Sonnet synth)",
    "[dim]~$0.005-0.02 per attempt[/dim]",
)
console.print(cost_table)


### Cell 16 — Receipts: write `engine/decisions/mcp-and-youtube.md`

In [ ]:
DECISIONS_PATH = Path("../engine/decisions/mcp-and-youtube.md")
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

receipts = f"""# MCP server + YouTube adapter receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/06_mcp_youtube_adapter.ipynb`
**Design refs:** `docs/marginalia-design.md` §5.1 (source layer), §7.4 (tool use), §8 (YouTube row), §9.6 (URL pattern dispatch).

## Tool Use: Messages API vs subprocess FastMCP

Both paths exercised on the same `list_files` + `read_file` tools, same
`raw/` inbox.

| Dimension | Messages API tool_use loop | Subprocess + stdio FastMCP |
|---|---|---|
| Wall time (smoke) | {agent_wall:.2f}s ({len(tool_calls)} tool calls + final answer) | {subprocess_wall*1000:.1f}ms (build_server) |
| IPC overhead | none (in-process Python calls) | stdio JSON-RPC |
| Client compatibility | any process using the Anthropic SDK | any MCP client |
| Code complexity | explicit loop with tool_use parsing | module + subprocess.Popen |
| Recommended when | tools are app-specific glue code | tools are shared across many clients |

The engine extract is the **subprocess version**
(`engine/adapters/local_fs/mcp_server.py`, runnable via
`uv run python -m engine.adapters.local_fs.mcp_server`). Reusable
beyond the Agent SDK, registrable in Claude Desktop / Cursor configs.
The Messages API tool_use loop stays in NB 06 as a learning artifact —
useful when an adapter is app-specific glue code (`marginalia` CLI
internals, model_comparison harness) or you want full visibility into
the agent loop.

## YouTube adapter — failure-mode coverage

Three failure paths returned structured `ExtractedContent` with
`failure_reason` populated, no exceptions:

1. Invalid URL (non-YouTube host) → `failure_reason="invalid_url: ..."`.
2. Empty transcript / cached_segments=[] → `failure_reason="empty_transcript"`.
3. Live `NoTranscriptFound` / `VideoUnavailable` / `TranscriptsDisabled` →
   `failure_reason="<api_error_class>: <message>"`.

This means batch operations (e.g. "ingest 50 channel videos") can
collect partial success without a giant try/except wrapper.

## Cost shape

| Path | Cost (USD) |
|---|---|
| Transcript fetch (cached fixture, no LLM) | $0.000000 |
| Haiku 4.5 summary on 3Blue1Brown (200-word budget) | ${result_with_summary.cost_usd:.6f} |
| Haiku 4.5 summary on synthetic CJK (400-word budget) | ${cjk_result.cost_usd:.6f} |

Latin-vs-CJK budget asymmetry: the 400-word CJK budget produces ~2×
the output tokens of the Latin path, but the input transcript is also
denser (CJK characters carry more semantic weight per char). Net cost
is similar order of magnitude.

## Selected design choices

| Decision | Choice | Rationale |
|---|---|---|
| Engine extract | Subprocess FastMCP server | Reusable across clients; matches §10G ("local-filesystem MCP server pointed at `~/wiki-raw/`"). |
| Adapter return type | `ExtractedContent` (extended) | One contract for all adapters; routes through `extract_url()` next to `extract(path)`. |
| Failure handling | Structured `failure_reason`, no raise | Batch-friendly; matches §7.3.1 "never silent failure" spirit at the adapter layer. |
| Summary model | Haiku 4.5 (default) | Cheap, fast, appropriate for narrow summarization task; §7.2 source-summarization row. |
| CJK detection | Unicode-block ratio > 0.30 → 400-word budget | Heuristic; tunable via `CJK_RATIO_THRESHOLD` constant. |
| URL routing entry point | New `engine.utils.dispatch.extract_url()` | File-path dispatch (`extract(path)`) stays type-clean; URLs are a different domain. |

## Caveats

- One real fixture (3Blue1Brown) plus one synthetic CJK fixture. Real
  Japanese / Chinese YouTube fixtures would be more honest but
  fragile (videos get deleted, captions change). Widen the
  sample if/when language coverage matters for production.
- The 200/400-word summary budgets are guidelines; Haiku tends to
  honor them within ~10%. Don't rely on exact word counts.
- The subprocess server has been smoke-tested but not stress-tested.
  Pre-flight check before running headless agents against it.
- Whisper-based transcription for uncaptioned videos is out of scope
  (NB 06 silently skips them with `failure_reason`).

## Re-running

```python
from pathlib import Path
import json, asyncio
from anthropic import Anthropic
from engine.adapters import extract_youtube

cached = json.loads(Path("notebooks/data/transcripts/aircAruvnKk.json").read_text())
client = Anthropic()
result = asyncio.run(extract_youtube(
    "https://youtu.be/aircAruvnKk",
    client=client,
    cached_segments=cached,
))
print(result.extraction_method, result.cost_usd)
```

To refresh fixtures: `uv run python notebooks/_ops/fetch_youtube_fixtures.py`.
"""

DECISIONS_PATH.write_text(receipts, encoding="utf-8")
print(f"wrote {DECISIONS_PATH.resolve()}  ({DECISIONS_PATH.stat().st_size} bytes)")


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `extract_youtube()`, `extract_video_id()`, CJK helpers | `engine/adapters/youtube/extractor.py` (extracted) |
| Subprocess FastMCP server | `engine/adapters/local_fs/mcp_server.py` (extracted) |
| URL dispatch (`extract_url()`) | `engine/utils/dispatch.py` (extended) |
| `ExtractedContent.failure_reason` field + `youtube_transcript` literal | `engine/adapters/_template/contract.py` (extended) |
| Versioned summary prompt | `engine/prompts/youtube_summary.md` (extracted) |
| Cell 16 receipts | `engine/decisions/mcp-and-youtube.md` (extracted) |

**Notebook-only (intentionally not extracted):**
- The in-process `@tool` declarations (cell 4) — Claude Agent SDK pattern; if extracted, lives in `engine/agents/`, not `engine/adapters/`.
- The subprocess JSON-RPC handshake (cell 6) — diagnostic only; production clients use the SDK's MCP client which handles handshake automatically.
- The cost summary table (cell 15) — receipts version lives in `mcp-and-youtube.md`.

**Fixtures committed at `notebooks/data/transcripts/`:**
- `aircAruvnKk.json` (3Blue1Brown, 61 segments) — regenerate via `uv run python notebooks/_ops/fetch_youtube_fixtures.py`.
- `synthetic-cjk.json` — hand-built; do not regenerate.

**`CACHE_VERSION` discipline:** not triggered by NB 06 (no edits to existing prompts). The new `youtube_summary.md` starts at v1; future edits must bump in lockstep with `CACHE_VERSION` once it lands in NB 10.

**Out of scope (deferred to NB 07):**
- `engine/prompts/orchestrator.md` (the notebook-plan listed it under NB 06 extracts; deferred to keep scope tight).
- `marginalia add <url>` CLI command — orchestrator territory.
- The 6 wiki-layer tools (`marginalia.search`, etc.) from §7.4.
